# Préprocessing commun

Préprocessing pour les 4 modèles : régression linéaire, DNN, SVMR, XGBoost.

## 1. Chargement

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Merge_final_3 (plus de fuite)
df = pd.read_csv("../data/final_metropole/Merge_final_3.csv", dtype={"code_dept": str})

# on range par département puis par semaine
df = df.sort_values(["code_dept", "date_semaine"])
df.shape

(19968, 82)

## 2. Colonnes qu'on enlève

In [ ]:
# les autres pathologies
autres_cibles = [
    "65plus_allergie_urgences", "0_14_allergie_urgences", "15_64_allergie_urgences",
    "0_14_asthme_urgences", "15_64_asthme_urgences", "TousAge_asthme_urgences", "65plus_asthme_urgences",
    "0an_bronchiolite_urgences",
    "TousAge_allergie_hospitalisation", "65plus_allergie_hospitalisation",
    "0_14_allergie_hospitalisation", "15_64_allergie_hospitalisation",
    "0_14_asthme_hospitalisation", "15_64_asthme_hospitalisation",
    "TousAge_asthme_hospitalisation", "65plus_asthme_hospitalisation",
    "0an_bronchiolite_hospitalisation",
]

#nombres de malades : on a les prévalences
effectifs = [
    "Ntop_Maladie_Respi_Chronique_tsage_9",
    "Ntop_Maladie_Respi_Chronique_0-14_9", "Ntop_Maladie_Respi_Chronique_15-64_9", "Ntop_Maladie_Respi_Chronique_65+_9",
    "Npop_Maladie_Respi_Chronique_0-14_9", "Npop_Maladie_Respi_Chronique_15-64_9", "Npop_Maladie_Respi_Chronique_65+_9",
]

# nom du département et annee_effectif
df = df.drop(columns=autres_cibles + effectifs + ["nom_dep", "annee_effectif"])

# on renomme la colonnequi donne la population
df = df.rename(columns={"Npop_Maladie_Respi_Chronique_tsage_9": "population"})

# tranformation de SOS Médecin en 0/1
df["sos_present"] = df["sos_present"].astype(int)
df.shape

(19968, 56)

In [ ]:
df.isna().sum().sum()

np.int64(0)

## 3. Mise en forme des semaines

In [ ]:
df["annee"] = df["date_semaine"].str[:4].astype(int)
df["semaine"] = df["date_semaine"].str[-2:].astype(int)

# saisonnalité
df["semaine_sin"] = np.sin(2 * np.pi * df["semaine"] / 52)
df["semaine_cos"] = np.cos(2 * np.pi * df["semaine"] / 52)

## 4. Cible et semaines d'avant

In [ ]:
taux = df.groupby("code_dept")["TousAge_allergie_urgences"]

# cible : taux de la semaine suivante par département
df["cible"] = taux.shift(-1)
df["date_cible"] = df.groupby("code_dept")["date_semaine"].shift(-1)

# shift 1 & 2
df["lag_1_semaine_avant"] = taux.shift(1)
df["lag_2_semaines_avant"] = taux.shift(2)
df["moyenne_passe_2w"] = (df["TousAge_allergie_urgences"] + df["lag_1_semaine_avant"]) / 2

# moyenne des 4 dernières semaines (pour référence)
df["moy4"] = taux.transform(lambda s: s.rolling(4).mean())

# on enlève les bordures vides (début 2021 sans semaines, fin 2024 sans cible)
df = df.dropna(subset=["cible", "lag_2_semaines_avant"])
df.shape

(19680, 66)

## 5. Train, validation, test

In [ ]:
# on découpe selon l'année de la cible
df["annee_cible"] = df["date_cible"].str[:4].astype(int)

df["jeu"] = "train"
df.loc[df["annee_cible"] == 2023, "jeu"] = "val"
df.loc[df["annee_cible"] == 2024, "jeu"] = "test"

df.groupby("jeu")["date_cible"].agg(["count", "min", "max"])

,count,min,max
jeu,,,
test,4992,2024-S01,2024-S52
train,9696,2021-S04,2022-S52
val,4992,2023-S01,2023-S52


## 6. Semaines de pic

In [ ]:
# pic : semaine dans les 10% les plus hautes de son département 
# en enlevant 2024 pour éviter une fuite de données
sans_test = df[df["jeu"] != "test"]
seuils = sans_test.groupby("code_dept")["cible"].quantile(0.90)

df["pic"] = (df["cible"] >= df["code_dept"].map(seuils)).astype(int)

df.groupby("jeu")["pic"].mean().round(3)

jeu
test     0.065
train    0.107
val      0.101
Name: pic, dtype: float64

## 7. Références (point de comparaison)

In [ ]:
test = df[df["jeu"] == "test"]

def scores(y, pred):
    return {"R2": round(r2_score(y, pred), 3),
            "RMSE": round(mean_squared_error(y, pred) ** 0.5, 1),
            "MAE": round(mean_absolute_error(y, pred), 1)}

# 1) la persistance
# 2) la moyenne 4 semaines précédentes
references = {
    "Persistance": scores(test["cible"], test["TousAge_allergie_urgences"]),
    "Moyenne des 4 dernières semaines": scores(test["cible"], test["moy4"]),
}
pd.DataFrame(references).T

,R2,RMSE,MAE
Persistance,0.035,287.9,206.7
Moyenne des 4 dernières semaines,0.349,236.5,171.6


## 8. Sauvegarde

In [10]:
df.to_csv("../data/final_metropole/donnees_modeles.csv", index=False)

## 9. À ajouter au début de chaque modèle

import pandas as pd

df = pd.read_csv("../data/final_metropole/donnees_modeles.csv", dtype={"code_dept": str})

pas_variables = ["date_semaine", "date_cible", "annee", "annee_cible", "semaine", "jeu",
                 "TousAge_allergie_urgences", "cible", "pic", "moy4", "population"]
variables = [c for c in df.columns if c not in pas_variables]

train = df[df["jeu"] == "train"]
val = df[df["jeu"] == "val"]
test = df[df["jeu"] == "test"]
train_val = df[df["jeu"] != "test"]

X_train, y_train = train[variables], train["cible"]
X_val, y_val = val[variables], val["cible"]
X_test, y_test = test[variables], test["cible"]
X_train_val, y_train_val = train_val[variables], train_val["cible"]
